<a href="https://colab.research.google.com/github/BensonLing0925/Cuda_Kernel_Learning/blob/main/Day2_tiled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Jun 20 08:01:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.9 MB/s eta 0:00:00


In [3]:
import torch
from torch.utils.cpp_extension import load_inline

cuda_src = r'''
__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
__global__ void matmul_naive(const float* A, const float* B, float* C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    // boundary checks
    if (row < M && col < N) {
      float sum = 0;
      for (int k = 0 ; k < K ; ++k) {
        sum += A[row*K+k]*B[k*N+col];
      }
      C[row*N+col]=sum;
    }
}
torch::Tensor vadd_cuda(torch::Tensor a, torch::Tensor b) {
    auto c = torch::empty_like(a);
    int n = a.numel(), threads = 256, blocks = (n + threads - 1) / threads;
    vadd<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), n);
    return c;
}
torch::Tensor matmul_naive_cuda(torch::Tensor a, torch::Tensor b) {
    int M = a.size(0), K = a.size(1), N = b.size(1);
    auto c = torch::empty({M, N}, a.options());
    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (M + 15) / 16);
    matmul_naive<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), M, N, K);
    return c;
}
'''
naive = load_inline(name="matmul_naive", cpp_sources="torch::Tensor matmul_naive_cuda(torch::Tensor, torch::Tensor);",
                  cuda_sources=cuda_src,extra_cuda_cflags=["-arch=sm_75"],functions=["matmul_naive_cuda"], verbose=True)

A = torch.randn(100, 100, device="cuda")
B = torch.randn(100, 100, device="cuda")
ref = A @ B
cuda_out = naive.matmul_naive_cuda(A, B)
ref = ref.to("cpu")
cuda_out = cuda_out.to("cpu")
print("max error:", (cuda_out - ref).abs().max().item())

max error: 1.52587890625e-05


In [4]:
import torch
from torch.utils.cpp_extension import load_inline

cuda_src = r'''
#define TILE 16

__global__ void matmul_tiled(const float* A, const float* B, float* C,
              int M, int N, int K) {
    __shared__ float As[TILE][TILE];  //using shared memory and not HBM
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.0f;

    for (int t = 0; t < (K + TILE - 1) / TILE; t++) {
        // Each thread loads one element into shared memory.
        As[threadIdx.y][threadIdx.x] = A[row * K + (t * TILE + threadIdx.x)];
        Bs[threadIdx.y][threadIdx.x] = B[col + (t * TILE + threadIdx.y) * N];

        __syncthreads();   // wait for loading all data in the block

        for (int k = 0; k < TILE; k++)
          acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();   // wait for all threads in the block
    }

    if (row < M && col < N)
      C[row * N + col] = acc;
}

torch::Tensor tiled_matmul_cuda(torch::Tensor a, torch::Tensor b) {
    int M = a.size(0), K = a.size(1), N = b.size(1);
    auto c = torch::empty({M, N}, a.options());
    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (M + 15) / 16);
    matmul_tiled<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), M, N, K);
    return c;
}
'''

tiled = load_inline(name="matmul_tiled", cpp_sources="torch::Tensor tiled_matmul_cuda(torch::Tensor, torch::Tensor);",
                  cuda_sources=cuda_src,extra_cuda_cflags=["-arch=sm_75"],functions=["tiled_matmul_cuda"], verbose=True)
M, K, N = 1024, 1024, 1024
A = torch.randn(M, K, device="cuda")
B = torch.randn(K, N, device="cuda")
ref = A @ B
cuda_out = tiled.tiled_matmul_cuda(A, B)
print("max error:", (cuda_out.cpu() - ref.cpu()).abs().max().item())

max error: 0.000213623046875


In [7]:
# benchmark
import torch
def benchmark(fn, A, B, iters=50, warmup=10):
  for _ in range(warmup):
    fn(A, B)
  torch.cuda.synchronize()
  start = torch.cuda.Event(enable_timing=True)
  end = torch.cuda.Event(enable_timing=True)
  start.record()
  for _ in range(iters):
    fn(A, B)
  end.record()
  torch.cuda.synchronize()
  ms = start.elapsed_time(end) / iters
  M, K = A.shape ; N = B.shape[1]
  gflops = (2 * M * N * K) / (ms * 1e-3) / 1e9
  return ms, gflops

M, K, N = 1024, 1024, 1024
A = torch.randn(M, K, device="cuda")
B = torch.randn(K, N, device="cuda")

ms_n, gf_n = benchmark(naive.matmul_naive_cuda, A, B)
ms_tiled, gf_tiled = benchmark(tiled.tiled_matmul_cuda, A, B)
ms_t, gf_t = benchmark(lambda a, b: a @ b, A, B)
print(f"naive : {ms_n:7.3f} ms   {gf_n:7.1f} GFLOP/s")
print(f"tiled : {ms_tiled:7.3f} ms   {gf_tiled:7.1f} GFLOP/s")
print(f"tiled is {ms_n/ms_tiled:.2f}x faster than naive")

naive :   6.277 ms     342.1 GFLOP/s
tiled :   2.746 ms     782.1 GFLOP/s
tiled is 2.29x faster than naive
